### Heart Failure Prediction Dataset

**Ссылка для скачивания:**  
<https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction/data>

### О датасете
- **918 записей**, 11 клинических признаков
- **Целевая переменная:** `HeartDisease` (1 — есть заболевание, 0 — нет)
- Объединяет **5 датасетов**: Cleveland, Hungarian, Switzerland, Long Beach VA, Stalog
- **Пропусков нет**, но в ходе EDA обнаружится другая проблема
- **Задача:** бинарная классификация для раннего выявления сердечно-сосудистых заболевани

### A) Быстрый обзор данных (Pandas)

In [2]:
#импорт бибилотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
from sklearn.preprocessing import StandardScaler, MinMaxScaler

#настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

#загрузка данных
df = pd.read_csv("heart.csv")

`df.head()`

In [3]:
df.head(10)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
5,39,M,NAP,120,339,0,Normal,170,N,0.0,Up,0
6,45,F,ATA,130,237,0,Normal,170,N,0.0,Up,0
7,54,M,ATA,110,208,0,Normal,142,N,0.0,Up,0
8,37,M,ASY,140,207,0,Normal,130,Y,1.5,Flat,1
9,48,F,ATA,120,284,0,Normal,120,N,0.0,Up,0


`df.tail()`

In [4]:
df.tail(10)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
908,63,M,ASY,140,187,0,LVH,144,Y,4.0,Up,1
909,63,F,ASY,124,197,0,Normal,136,Y,0.0,Flat,1
910,41,M,ATA,120,157,0,Normal,182,N,0.0,Up,0
911,59,M,ASY,164,176,1,LVH,90,N,1.0,Flat,1
912,57,F,ASY,140,241,0,Normal,123,Y,0.2,Flat,1
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1
917,38,M,NAP,138,175,0,Normal,173,N,0.0,Up,0


`df.shape`

In [5]:
df.shape

(918, 12)

Общая информация о датасете\
`df.info()`

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 86.2 KB


`df.describe()`

In [7]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


Отдельно `df.describe(include="object")`

In [8]:
df.describe(include=["object", "string"])

,Sex,ChestPainType,RestingECG,ExerciseAngina,ST_Slope
count,918,918,918,918,918
unique,2,4,3,2,3
top,M,ASY,Normal,N,Flat
freq,725,496,552,547,460


`isnull().sum()`

In [9]:
df.isnull().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

### B) Пропуски и очистка

Значение **min** показателей **RestingBP** (Артериальное давление в покое) и **Cholesterol** (Холестерин) равны 0, что невозмнонжно, посчитаем количество таких колонок

In [10]:
print("RestingBP zeros:", (df["RestingBP"] == 0).sum())
print("Cholesterol zeros:", (df["Cholesterol"] == 0).sum())

RestingBP zeros: 1
Cholesterol zeros: 172


Так как имеется только одна запись содержит **RestingBP = 0**, её можно спокойно удалить, так как это лишь 1 из 918 записей

In [11]:
df = df[df["RestingBP"] > 0]

Количесво записей с **Cholesterol = 0** равно 172, что уже является замтеной долей в данных, 172/917 = 18.75%, просто так удалить их нельзя.\
Посмотрим распределение нулей по нашему целевому признаку **HeartDisease** чтобы понять, есть ли между ними какая-то связь

In [12]:
print("Распределение нулей по целевому признаку:")
print(df.groupby('HeartDisease')['Cholesterol'].apply(lambda x: (x==0).sum()))

Распределение нулей по целевому признаку:
HeartDisease
0     20
1    151
Name: Cholesterol, dtype: int64


Видим, что 151/172 = 87.7% нулей у пациентов с обнаруженными сердечными заболеваниями. Созданим новый признак **Cholesterol_missing** который будет указывать, равно ли значение **Cholesterol** нулю

In [13]:
df["Cholesterol_missing"] = (df["Cholesterol"] == 0).astype(int)

In [14]:
median_ill = df[df["HeartDisease"] == 1]["Cholesterol"].median()
median_healty = df[df["HeartDisease"] == 0]["Cholesterol"].median()

print(f"Медиана холестерина у больных: {median_ill:.0f}")
print(f"Медиана холестерина у здоровых: {median_healty:.0f}")

Медиана холестерина у больных: 217
Медиана холестерина у здоровых: 227


Заполняем нули раздельно, потому что у 151 больного нет холестерина, а у здоровых только 20 - если всем поставить общую медиану, мы больным завысим показатель, а здоровым занизим, и модель запутается. Поэтому больным с пропуском ставим медиану холестерина больных, здоровым с пропуском - медиану здоровых. Флаг **Cholesterol_missing** нужен, чтобы показать модели, что значение восстановленное, а не реальное - сам факт пропуска (особенно у больных) тоже важная информация.

In [15]:
df.loc[(df['Cholesterol'] == 0) & (df['HeartDisease'] == 1), 'Cholesterol'] = median_ill
df.loc[(df['Cholesterol'] == 0) & (df['HeartDisease'] == 0), 'Cholesterol'] = median_healty

`df.describe()` после проиведенных манипуляций

In [16]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Cholesterol_missing
count,917.000000,917.000000,917.000000,917.000000,917.000000,917.000000,917.000000,917.000000
mean,53.509269,132.540894,239.700109,0.233370,136.789531,0.886696,0.552890,0.186478
std,9.437636,17.999749,54.352727,0.423206,25.467129,1.066960,0.497466,0.389704
min,28.000000,80.000000,85.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,47.000000,120.000000,214.000000,0.000000,120.000000,0.000000,0.000000,0.000000
50%,54.000000,130.000000,225.000000,0.000000,138.000000,0.600000,1.000000,0.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000,0.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000,1.000000


### C) Расширенная статистика
Для числовых колонок выведем: min, max, mean, median, mode\
Посчитаем quantile.

In [17]:
num_cols = df.select_dtypes(include=[np.number]).columns

stats = df[num_cols].describe().T

for col in num_cols:
    stats.loc[col, "mode"] = df[col].mode()[0]

for col in num_cols:
    stats.loc[col, "5%"] = df[col].quantile(0.05)
    stats.loc[col, "95%"] = df[col].quantile(0.95)

stats = stats[['min', '5%', '25%', '50%', '75%', '95%', 'max', 'mean', 'mode']]
stats.columns = ['min', '5%', '25%', 'median', '75%', '95%', 'max', 'mean', 'mode']

print(stats.round(2))

                      min     5%    25%  median    75%    95%    max    mean   mode
Age                  28.0   37.0   47.0    54.0   60.0   68.0   77.0   53.51   54.0
RestingBP            80.0  107.6  120.0   130.0  140.0  160.0  200.0  132.54  120.0
Cholesterol          85.0  168.0  214.0   225.0  267.0  331.4  603.0  239.70  217.0
FastingBS             0.0    0.0    0.0     0.0    0.0    1.0    1.0    0.23    0.0
MaxHR                60.0   96.0  120.0   138.0  156.0  178.0  202.0  136.79  150.0
Oldpeak              -2.6    0.0    0.0     0.6    1.5    3.0    6.2    0.89    0.0
HeartDisease          0.0    0.0    0.0     1.0    1.0    1.0    1.0    0.55    1.0
Cholesterol_missing   0.0    0.0    0.0     0.0    0.0    1.0    1.0    0.19    0.0


Посчитаем дисперсию (variance), асимметрию (skewness) и эксцесс (kurtosis)

In [ ]:
from scipy.stats import skew, kurtosis

stats_advanced = pd.DataFrame(index=num_cols)

for col in num_cols:
    stats_advanced.loc[col, 'Variance'] = df[col].var()
    stats_advanced.loc[col, 'Skewness'] = skew(df[col])
    stats_advanced.loc[col, 'Kurtosis'] = kurtosis(df[col])

print(stats_advanced.round(2))

                     Variance  Skewness  Kurtosis
Age                     89.07     -0.20     -0.39
RestingBP              323.99      0.61      0.78
Cholesterol           2954.22      1.55      6.07
FastingBS                0.18      1.26     -0.41
MaxHR                  648.57     -0.14     -0.45
Oldpeak                  1.14      1.02      1.19
HeartDisease             0.25     -0.21     -1.95
Cholesterol_missing      0.15      1.61      0.59


**Что это значит** для наших данный?\
**Age (Возраст):**
- Skewness (-0.20): Близка к нулю, отрицательная. Распределение почти симметричное, лишь слегка смещенное влево, *mean* меньше *median* (возможно, немного больше молодых пациентов).
- Kurtosis (-0.39): Близка к нулю, отрицательная. Распределение возраста очень близко к нормальном, немного более плоское, чем идеальный колокол, выбросов нет.
**RestingBP (Артериальное давление в покое):**